# UC-JL-1 — Copy a Platform Notebook and Edit It

**Persona:** Scientist

**Goal:** Discover a published platform notebook by tag, copy it to a personal tenant
namespace, inspect the `INIT_CONTEXT` injection, add a new cell, save the edited version,
and verify the change persisted.

**Prerequisites:**
- JupyterLite Pyodide httpx fix committed: `window.location.origin + API_BASE` written
  into `INIT_CONTEXT` at serve time; `.ipynb` seeds reference `DYNASTORE_BASE_URL` env var
- At least one platform notebook tagged `ndvi` published by a sysadmin
- `BUILD_JUPYTERLITE=true` (or `false` — the REST API works either way; only the browser
  IDE requires the static assets)

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`, `CATALOG_ID`

In [ ]:
import os
import json
import copy

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = os.environ.get("DYNASTORE_TOKEN", "")
CATALOG_ID = os.environ.get("CATALOG_ID", "demo-catalog")

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0)
print(f"Connected to {BASE_URL}  catalog_id={CATALOG_ID}")

In [ ]:
# List platform notebooks tagged 'ndvi'
r = client.get("/notebooks/platform", params={"tags": "ndvi", "limit": 10})
print(r.status_code)
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

body = r.json()
notebooks = body if isinstance(body, list) else body.get("notebooks", body.get("items", []))
assert len(notebooks) > 0, "No platform notebooks with tag 'ndvi' found; publish one first."

for nb in notebooks:
    print(f"  id={nb['id']}  title={nb.get('title', nb.get('metadata', {}).get('title', '?'))}")

platform_notebook_id = notebooks[0]["id"]
print(f"\nUsing platform_notebook_id={platform_notebook_id}")

In [ ]:
# Copy the platform notebook into the tenant namespace
r = client.post(f"/notebooks/{CATALOG_ID}/copy/{platform_notebook_id}")
print(r.status_code, r.text[:400])
assert r.status_code == 201, f"Expected 201, got {r.status_code}: {r.text}"

copy_resp = r.json()
notebook_code = copy_resp["id"]
print(f"Notebook copied successfully.  notebook_code={notebook_code}")

In [ ]:
# Fetch the copy and inspect the INIT_CONTEXT cell
r = client.get(f"/notebooks/{CATALOG_ID}/{notebook_code}")
print(r.status_code)
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

notebook_dict = r.json()
cells = notebook_dict.get("cells", [])
print(f"Cell count: {len(cells)}")

# Locate the INIT_CONTEXT injection cell
init_cell = None
for cell in cells:
    src = "".join(cell.get("source", []))
    if "DYNASTORE_BASE_URL" in src or "INIT_CONTEXT" in src:
        init_cell = cell
        break

if init_cell:
    print("\nINIT_CONTEXT cell found (cell_type=%s):" % init_cell["cell_type"])
    print("".join(init_cell["source"])[:600])
else:
    print("WARNING: no INIT_CONTEXT / DYNASTORE_BASE_URL cell found in notebook.")
    print("Verify that the .ipynb seed or platform notebook includes the injection pattern.")

In [ ]:
# Edit notebook: append a new markdown cell, then PUT the updated notebook back
edited_notebook = copy.deepcopy(notebook_dict)

new_cell = {
    "id": "9f8e7d6c",
    "cell_type": "markdown",
    "metadata": {},
    "source": ["## Analyst Notes\n", "\n", "NDVI threshold adjusted to 0.35 for arid zones.\n"],
}
edited_notebook["cells"].append(new_cell)

r = client.put(
    f"/notebooks/{CATALOG_ID}/{notebook_code}",
    content=json.dumps(edited_notebook),
)
print(r.status_code, r.text[:300])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
print("Notebook updated successfully.")

In [ ]:
# Re-fetch and verify the new cell was persisted
r = client.get(f"/notebooks/{CATALOG_ID}/{notebook_code}")
print(r.status_code)
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

saved = r.json()
saved_cells = saved.get("cells", [])
assert len(saved_cells) == len(edited_notebook["cells"]), (
    f"Cell count mismatch: expected {len(edited_notebook['cells'])}, got {len(saved_cells)}"
)

last_src = "".join(saved_cells[-1].get("source", []))
assert "Analyst Notes" in last_src, f"New cell not found in saved notebook. last cell: {last_src!r}"
print(f"Persistence verified. Total cells: {len(saved_cells)}")
print(f"Last cell source: {last_src[:120]}")

## Edge Cases

The next three cells document known platform footguns. They are **informational** and do
not mutate notebook state.

In [ ]:
# Edge case 1: BUILD_JUPYTERLITE=false
# When the platform is deployed without BUILD_JUPYTERLITE=true, the static JupyterLite
# assets are not embedded. The /notebooks REST API still works (CRUD, copy, list),
# but the browser-based IDE at /jupyterlite/ returns 404.
#
# Symptom:  GET /jupyterlite/index.html  ->  404 Not Found
# Diagnosis: check the Cloud Run service env var BUILD_JUPYTERLITE.
# Fix:       redeploy with BUILD_JUPYTERLITE=true and confirm the static dir is present.
#
# This cell documents the footgun; it does not call an endpoint.
print(
    "BUILD_JUPYTERLITE=false: REST API works, browser IDE unavailable.\n"
    "Set BUILD_JUPYTERLITE=true and redeploy to enable the embedded JupyterLite UI."
)

In [ ]:
# Edge case 2: DYNASTORE_BASE_URL missing from .ipynb seed (broken-proxy footgun)
#
# When a seed notebook hardcodes BASE_URL = "http://localhost:8080" the Pyodide kernel
# inside the browser sends requests to localhost, which always fails behind a proxy.
#
# Correct pattern (injected by the platform at serve time via INIT_CONTEXT):
#
#   import js  # Pyodide only
#   BASE_URL = js.window.location.origin + "/api"
#
# Or, when running outside the browser (CI, local Python):
#
#   BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
#
# The platform writes the correct value into INIT_CONTEXT so both paths resolve
# without manual edits. If DYNASTORE_BASE_URL is absent from INIT_CONTEXT, httpx
# calls in Pyodide will hit 'Failed to fetch' with no helpful error message.
print(
    "Broken-proxy footgun: hardcoded localhost in seed notebook breaks all Pyodide httpx calls.\n"
    "Fix: use window.location.origin pattern via INIT_CONTEXT injection."
)

In [ ]:
# Edge case 3: Pyodide wheel limitation (disablePyPIFallback)
#
# JupyterLite with Pyodide only supports packages available in the Pyodide distribution
# or pre-built wheels. Packages that require C-extension compilation (e.g. psycopg2,
# cryptography) are not available unless a Pyodide-compatible wheel is provided.
#
# The platform sets disablePyPIFallback: true in jupyter-config-data to prevent
# confusing 'Connection refused' errors from pip falling back to PyPI.
#
# Attempting to install an unlisted package raises a clear error:
#   MicropipError: 'psycopg2' is not available in Pyodide or as a pure-Python wheel.
#
# Verify the behaviour by attempting an unsupported import (documentation only):
try:
    import micropip  # type: ignore[import]  # Pyodide-only module
    # In a real Pyodide cell this would raise MicropipError for unsupported packages.
    print("micropip available (running inside Pyodide).")
except ImportError:
    print(
        "micropip not available (running outside Pyodide — standard CPython).\n"
        "In the JupyterLite browser IDE, unsupported packages raise MicropipError.\n"
        "Only httpx, pydantic, and standard-library packages are safe to use."
    )

## Teardown

Delete the tenant copy of the notebook. The original platform notebook is unaffected.

In [ ]:
r = client.delete(f"/notebooks/{CATALOG_ID}/{notebook_code}")
print(r.status_code, r.text[:200])
assert r.status_code == 204, f"Expected 204, got {r.status_code}: {r.text}"
print(f"Notebook {notebook_code} deleted from catalog {CATALOG_ID}.")
client.close()